In [1]:
import pandas as pd

# 1. Caricamento del dataset scaricato da Kaggle
df = pd.read_csv("games.csv")
print(f"Partite totali nel dataset originale: {len(df)}")

# 2. LA MODIFICA: Rimuoviamo i pareggi per un modello binario puro
df = df[df['winner'] != 'draw'].copy()

# 3. Selezione delle colonne che ci interessano e pulizia dei nulli
colonne_utili = ['moves', 'white_rating', 'black_rating', 'winner']
df = df[colonne_utili].copy()
df = df.dropna()

# 4. Creazione Colonna Target e Feature Iniziale
# Ora abbiamo solo 'white' e 'black'. 
# 1 = Vittoria Bianco, 0 = Vittoria Nero
df['Vittoria_Bianco'] = (df['winner'] == 'white').astype(int)

# Calcoliamo il divario di bravura (fondamentale per aiutare il modello)
df['Differenza_Elo'] = df['white_rating'] - df['black_rating']

# 5. Filtro per le partite arrivate almeno alla mossa 20 (40 ply)
# Contiamo gli spazi nella stringa 'moves' per capire quanto è durata
df['numero_mezze_mosse'] = df['moves'].apply(lambda x: len(str(x).split()))
df_filtrato = df[df['numero_mezze_mosse'] >= 40].copy()

# 6. Salviamo solo l'essenziale per il prossimo script
df_finale = df_filtrato[['moves', 'Differenza_Elo', 'Vittoria_Bianco']].copy()

print(f"Partite valide per l'addestramento: {len(df_finale)}")
# Visualizziamo le prime 5 righe
print(df_finale.head())

Partite totali nel dataset originale: 20058
Partite valide per l'addestramento: 13616
                                               moves  Differenza_Elo  \
2  e4 e5 d3 d6 Be3 c6 Be2 b5 Nd2 a5 a4 c5 axb5 Nc...              -4   
3  d4 d5 Nf3 Bf5 Nc3 Nf6 Bf4 Ng4 e3 Nc6 Be2 Qd7 O...             -15   
4  e4 e5 Nf3 d6 d4 Nc6 d5 Nb4 a3 Na6 Nc3 Be7 b4 N...              54   
8  e4 e5 Bc4 Nc6 Nf3 Nd4 d3 Nxf3+ Qxf3 Nf6 h3 Bc5...              47   
9  e4 d5 exd5 Qxd5 Nc3 Qe5+ Be2 Na6 d4 Qf5 Bxa6 b...             172   

   Vittoria_Bianco  
2                1  
3                1  
4                1  
8                0  
9                1  


In [ ]:
import pandas as pd
import chess

# Dizionario con il numero esatto di pezzi a inizio partita
# Maiuscole = Bianco, Minuscole = Nero
pezzi_standard = {
    'P': 8, 'N': 2, 'B': 2, 'R': 2, 'Q': 1, 'K': 1,
    'p': 8, 'n': 2, 'b': 2, 'r': 2, 'q': 1, 'k': 1
}

def estrai_coordinate(mosse_str):
    """
    Gioca la partita fino alla mossa 20 (40 ply) ed estrae le coordinate X,Y.
    """
    board = chess.Board()
    mosse = str(mosse_str).split()
    
    # Facciamo avanzare la partita di esattamente 40 mezze-mosse
    for mossa in mosse[:40]:
        try:
            board.push_san(mossa)
        except:
            break
            
    # Contatori per tracciare quanti pezzi di ogni tipo troviamo
    trovati = {pezzo: 0 for pezzo in pezzi_standard.keys()}
    coord_dict = {}
    
    # Scansioniamo tutte le 64 case della scacchiera
    for square in chess.SQUARES:
        pezzo = board.piece_at(square)
        if pezzo:
            simbolo = pezzo.symbol()
            trovati[simbolo] += 1
            indice = trovati[simbolo]
            
            # Calcoliamo X (colonna) e Y (riga) da 1 a 8
            # La libreria ci dà metodi comodi per file (0-7) e rank (0-7)
            x = chess.square_file(square) + 1
            y = chess.square_rank(square) + 1
            
            coord_dict[f"{simbolo}_{indice}_X"] = x
            coord_dict[f"{simbolo}_{indice}_Y"] = y
            
    # GESTIONE DEI PEZZI CATTURATI (Trucco di Navid)
    # Se abbiamo trovato meno pezzi del previsto, assegniamo (0,0) ai restanti
    for simbolo, max_count in pezzi_standard.items():
        count_trovati = trovati[simbolo]
        for i in range(count_trovati + 1, max_count + 1):
            coord_dict[f"{simbolo}_{i}_X"] = 0
            coord_dict[f"{simbolo}_{i}_Y"] = 0
            
    return coord_dict

print("Estrazione delle coordinate in corso... (potrebbe volerci qualche decina di secondi)")

# 1. Applichiamo la funzione a tutte le righe del dataframe della Fase 1
lista_dizionari = df_finale['moves'].apply(estrai_coordinate).tolist()

# 2. Convertiamo la lista di dizionari in un nuovo DataFrame Pandas
df_coordinate = pd.DataFrame(lista_dizionari)

# 3. Uniamo le coordinate con la Differenza Elo e il Target
# (Resettiamo l'indice per assicurarci che si uniscano perfettamente riga per riga)
df_finale = df_finale.reset_index(drop=True)
df_ml = pd.concat([df_finale[['Differenza_Elo', 'Vittoria_Bianco']], df_coordinate], axis=1)

print("Dataset per il Machine Learning creato con successo!")
print(f"Dimensioni del dataset: {df_ml.shape[0]} righe e {df_ml.shape[1]} colonne.")
print(df_ml.head())